# Entrenamiento YOLO26 con etiquetas revisadas en CVAT

**Entrada:** los lotes de `03_anotacion_cvat.ipynb` → `data/malaga/anotacion/<lote>/indice.csv` (solo filas con `estado = revisada`) + `labels/*.txt` + `images/`.

A diferencia de la versión anterior, **no se regeneran etiquetas con un detector**: las cajas son las revisadas por humanos, por lo que las métricas miden acierto frente a la realidad y no frente al modelo preentrenado (se evita el entrenamiento circular).

Diseño experimental:
1. **Test por cámaras no vistas**: un `TEST_RATIO` de las cámaras se aparta antes de todo y solo se usa al final.
2. **Validación cruzada por cámara** (`GroupKFold`) sobre el resto: las imágenes de una cámara nunca están a la vez en train y val, evitando fuga de información por escenario.
3. **Modelo:** YOLO26 (Ultralytics ≥ 8.4, arquitectura end-to-end sin NMS) partiendo de pesos COCO.
4. **Métricas:** globales (mAP50-95, mAP50, precisión, recall) y **por clase**, con detalle de los semáforos por color.

## 1. Configuración

In [ ]:
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from sklearn.model_selection import GroupKFold, train_test_split
from ultralytics import YOLO


PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import load_config, path
from deteccion import CLASS_NAMES, TRAFFIC_LIGHT_CLASSES, PEDESTRIAN_LIGHT_CLASSES

CFG = load_config()["entrenamiento"]
ANOTACION_DIR = path("anotacion")
DATASET_ROOT = path("dataset_yolo")
RESULTS_DIR = path("resultados")
RUNS_DIR = path("runs")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Modelo y entrenamiento (config.yaml -> entrenamiento)
MODEL_WEIGHTS = CFG["modelo"]  # yolo26n/s/m/l/x.pt (pesos COCO)
EPOCHS = CFG["epochs"]
PATIENCE = CFG["patience"]     # parada temprana si val no mejora
IMG_SIZE = CFG["imgsz"]        # las imágenes son 640x480; 960 puede ayudar con semáforos pequeños (más memoria)
BATCH = CFG["batch"]           # GTX 1650 (4 GB): 8 con yolo26s a 640
DEVICE = 0 if torch.cuda.is_available() else "cpu"
WORKERS = CFG["workers"]
RUN_NAME = "yolo26_malaga_cv"

# Particiones
N_FOLDS = CFG["n_folds"]
TEST_RATIO = CFG["test_ratio"]
RANDOM_STATE = CFG["semilla"]
MAX_FOLDS_TRAIN = None  # p.ej. 1 para una prueba rápida; None = todos

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Dispositivo:", torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU")
print("Clases:", dict(enumerate(CLASS_NAMES)))

## 2. Carga de los lotes anotados

In [ ]:
lotes = []
for indice_file in sorted(ANOTACION_DIR.glob("*/indice.csv")):
    df = pd.read_csv(indice_file, dtype={"camera": str})
    df["image_path"] = [indice_file.parent / "images" / name for name in df["image"]]
    df["label_file"] = [indice_file.parent / str(p) if isinstance(p, str) and p else None for p in df["label_path"]]
    lotes.append(df)

if not lotes:
    raise FileNotFoundError(f"No hay lotes en {ANOTACION_DIR}: ejecuta antes 03_anotacion_cvat.ipynb")

anotaciones = pd.concat(lotes, ignore_index=True)
print("Imágenes por lote y estado:")
display(anotaciones.groupby(["lote", "estado"]).size().unstack(fill_value=0))

frames_df = anotaciones[anotaciones["estado"] == "revisada"].copy()
missing = ~frames_df["image_path"].map(Path.exists) | ~frames_df["label_file"].map(lambda p: p is not None and Path(p).exists())
if missing.any():
    print(f"Aviso: {missing.sum()} imágenes revisadas sin imagen o etiqueta en disco (se descartan)")
frames_df = frames_df[~missing].reset_index(drop=True)

print(f"Imágenes revisadas: {len(frames_df)} | cámaras: {frames_df['camera'].nunique()} | cajas: {int(frames_df['n_cajas'].sum())}")

## 3. Partición: test por cámaras + validación cruzada por cámara (GroupKFold)

In [ ]:
cams = np.array(sorted(frames_df["camera"].unique()))
cams_trainval, cams_test = train_test_split(cams, test_size=TEST_RATIO, random_state=RANDOM_STATE)
df_trainval = frames_df[frames_df["camera"].isin(cams_trainval)].reset_index(drop=True)
df_test = frames_df[frames_df["camera"].isin(cams_test)].reset_index(drop=True)

N_FOLDS = min(N_FOLDS, len(cams_trainval))
folds = list(GroupKFold(n_splits=N_FOLDS).split(df_trainval, groups=df_trainval["camera"]))

print(f"Train+Val: {len(df_trainval)} imágenes ({len(cams_trainval)} cámaras)")
print(f"Test:      {len(df_test)} imágenes ({len(cams_test)} cámaras): {sorted(cams_test)}")
for i, (tr, va) in enumerate(folds):
    print(f"  Fold {i}: train={len(tr)} img/{df_trainval.loc[tr, 'camera'].nunique()} cám | "
          f"val={len(va)} img/{df_trainval.loc[va, 'camera'].nunique()} cám")


def class_counts(df):
    counts = pd.Series(0, index=CLASS_NAMES)
    for label_file in df["label_file"]:
        for line in Path(label_file).read_text(encoding="utf-8").splitlines():
            counts.iloc[int(line.split()[0])] += 1
    return counts


display(pd.DataFrame({"trainval": class_counts(df_trainval), "test": class_counts(df_test)}))

## 4. Estructura de dataset YOLO por fold

Se copian imágenes y etiquetas revisadas (sin re-inferir nada) a `yolo26_dataset/<fold>/{images,labels}/{train,val}`.

In [ ]:
def write_fold_dataset(fold_dir, splits):
    """splits: dict nombre_split -> DataFrame. Copia imágenes y etiquetas y genera dataset.yaml."""
    if fold_dir.exists():
        shutil.rmtree(fold_dir)
    for split, df_split in splits.items():
        img_out = fold_dir / "images" / split
        lbl_out = fold_dir / "labels" / split
        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)
        for row in df_split.itertuples():
            shutil.copy2(row.image_path, img_out / Path(row.image_path).name)
            shutil.copy2(row.label_file, lbl_out / f"{Path(row.image_path).stem}.txt")

    yaml_data = {
        "path": str(fold_dir),
        "train": "images/train",
        "val": "images/val",
        "names": dict(enumerate(CLASS_NAMES)),
    }
    yaml_path = fold_dir / "dataset.yaml"
    yaml_path.write_text(yaml.safe_dump(yaml_data, sort_keys=False, allow_unicode=True), encoding="utf-8")
    return yaml_path


def per_class_metrics(metrics):
    """mAP por clase. box.maps tiene una entrada por clase; ap50/p/r solo para las clases presentes (ap_class_index)."""
    df = pd.DataFrame({"clase": CLASS_NAMES, "mAP50-95": [float(v) for v in metrics.box.maps]})
    present = pd.DataFrame({
        "clase": [CLASS_NAMES[i] for i in metrics.box.ap_class_index],
        "mAP50": metrics.box.ap50,
        "precision": metrics.box.p,
        "recall": metrics.box.r,
    })
    df = df.merge(present, on="clase", how="left")
    df.loc[df["mAP50"].isna(), "mAP50-95"] = np.nan  # clase sin instancias en este split
    return df


print("Funciones definidas.")

## 5. Entrenamiento con validación cruzada

In [ ]:
cv_results = []
fold_yaml_paths = []

for fold_i, (tr_idx, va_idx) in enumerate(folds):
    if MAX_FOLDS_TRAIN is not None and fold_i >= MAX_FOLDS_TRAIN:
        break
    print(f"\n===== FOLD {fold_i} =====")
    yaml_path = write_fold_dataset(DATASET_ROOT / f"fold_{fold_i}", {
        "train": df_trainval.iloc[tr_idx],
        "val": df_trainval.iloc[va_idx],
    })
    fold_yaml_paths.append(yaml_path)

    model = YOLO(MODEL_WEIGHTS)
    model.train(
        data=str(yaml_path),
        epochs=EPOCHS,
        patience=PATIENCE,
        imgsz=IMG_SIZE,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,
        project=str(RUNS_DIR),
        name=f"{RUN_NAME}_fold{fold_i}",
        exist_ok=True,
        seed=RANDOM_STATE,
        pretrained=True,
        verbose=False,
    )

    best = YOLO(RUNS_DIR / f"{RUN_NAME}_fold{fold_i}" / "weights" / "best.pt")
    m = best.val(data=str(yaml_path), split="val", imgsz=IMG_SIZE, device=DEVICE, verbose=False)
    cv_results.append({
        "fold": fold_i,
        "mAP50-95": float(m.box.map),
        "mAP50": float(m.box.map50),
        "precision": float(m.box.mp),
        "recall": float(m.box.mr),
    })
    print(f"Fold {fold_i}: mAP50-95={cv_results[-1]['mAP50-95']:.4f} | mAP50={cv_results[-1]['mAP50']:.4f}")

cv_df = pd.DataFrame(cv_results)
print("\n===== RESUMEN CV =====")
display(cv_df)
print("\nMedia ± std:")
display(cv_df[["mAP50-95", "mAP50", "precision", "recall"]].agg(["mean", "std"]))

## 6. Métricas por clase y por color de semáforo (mejor fold)

In [ ]:
if not cv_results:
    print("No hay resultados de CV. Ejecuta el entrenamiento primero.")
else:
    best_fold = int(cv_df.loc[cv_df["mAP50-95"].idxmax(), "fold"])
    print(f"Mejor fold: {best_fold}")

    best_model = YOLO(RUNS_DIR / f"{RUN_NAME}_fold{best_fold}" / "weights" / "best.pt")
    m = best_model.val(data=str(fold_yaml_paths[best_fold]), split="val", imgsz=IMG_SIZE, device=DEVICE, verbose=False)

    map_df = per_class_metrics(m)
    print("\n=== Métricas por clase ===")
    display(map_df)

    tl_df = map_df[map_df["clase"].isin(TRAFFIC_LIGHT_CLASSES + PEDESTRIAN_LIGHT_CLASSES)].dropna(subset=["mAP50"])
    if not tl_df.empty:
        fig, ax = plt.subplots(figsize=(8, 4))
        x = np.arange(len(tl_df))
        ax.bar(x - 0.2, tl_df["mAP50-95"], width=0.4, label="mAP50-95")
        ax.bar(x + 0.2, tl_df["mAP50"], width=0.4, label="mAP50")
        ax.set_xticks(x)
        ax.set_xticklabels(tl_df["clase"], rotation=20)
        ax.set_ylim(0, 1)
        ax.set_ylabel("mAP")
        ax.set_title("mAP de semáforos por color (mejor fold)")
        ax.legend()
        ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.show()

    map_df.to_csv(RESULTS_DIR / "metricas_map_por_clase.csv", index=False)
    print(f"Métricas guardadas en: {RESULTS_DIR / 'metricas_map_por_clase.csv'}")

## 7. Evaluación final sobre test (cámaras no vistas)

In [ ]:
if not cv_results:
    print("No hay modelo entrenado.")
else:
    test_yaml = write_fold_dataset(DATASET_ROOT / "test_holdout", {"train": df_test.iloc[:0], "val": df_test})
    test_metrics = best_model.val(data=str(test_yaml), split="val", imgsz=IMG_SIZE, device=DEVICE, verbose=False)
    print("=== Test holdout (cámaras no vistas en train/val) ===")
    print(f"mAP50-95:  {float(test_metrics.box.map):.4f}")
    print(f"mAP50:     {float(test_metrics.box.map50):.4f}")
    print(f"Precisión: {float(test_metrics.box.mp):.4f}")
    print(f"Recall:    {float(test_metrics.box.mr):.4f}")
    print(f"Inferencia: {test_metrics.speed['inference']:.1f} ms/imagen")

    test_df = per_class_metrics(test_metrics)
    display(test_df)
    test_df.to_csv(RESULTS_DIR / "metricas_test_por_clase.csv", index=False)

In [ ]:
# Export opcional para despliegue
# best_model.export(format="onnx")
print("Export comentado por defecto. Descomenta si lo necesitas.")